# Stage 2 - Instruction Fine-Tuning (SFT)
### GenAI / Agentic AI Domain Assistant &middot; Base model: `Qwen2.5-0.5B` &middot; Framework: [Unsloth](https://github.com/unslothai/unsloth)

Pipeline: Base Model &rarr; Stage 1: Non-Instruction FT &rarr; **Stage 2: Instruction FT** &rarr; Stage 3: DPO/ORPO Alignment

This notebook:
1. Loads the Stage 1 (`models/non_instruction_adapter/`) checkpoint as the starting point
2. Formats `data/instruction_dataset.jsonl` (101 Q&A pairs) with a ChatML template
3. Applies LoRA and runs supervised instruction fine-tuning, masking loss to assistant turns only
4. Saves `models/sft_adapter/`
5. Re-evaluates on the same 10 questions vs. the base model &rarr; writes `reports/sft_model_comparison.md`

> Run on a free Colab/Kaggle T4 GPU. Total runtime: ~5-10 minutes for this 0.5B model.


In [ ]:
%%capture
# Colab: Runtime > Change runtime type > T4 GPU, then run this cell first.
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U "trl>=0.12.0" peft accelerate bitsandbytes datasets sentencepiece


In [ ]:
import os

REPO_URL = "https://github.com/Abhishek15016/prepminds.git"

if not os.path.exists("data") and REPO_URL:
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not os.path.exists(repo_name):
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(repo_name)

assert os.path.exists("data"), "Could not find data/ - clone/open the repo first."
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
import sys
sys.path.insert(0, os.getcwd())


In [ ]:
from src.eval_questions import EVAL_QUESTIONS, judge_pair, markdown_table

print(f"Loaded {len(EVAL_QUESTIONS)} canonical evaluation questions (same set used in every stage).")


## Step 0 - (Optional) Hugging Face Hub setup

Lets this notebook automatically pull the Stage 1 adapter from the Hub if it isn't on local disk (e.g. this is a fresh Colab session from the one that ran Stage 1), and push this stage's adapter when done.

In [ ]:
from huggingface_hub import login
import os

# --- Hugging Face Hub config -------------------------------------------------
# Colab wipes local disk between sessions, so each notebook can optionally push
# its adapter to the Hub, and the *next* notebook will automatically pull it
# back down if no local copy is found. Set your HF username below and
# (recommended) add a Colab secret named HF_TOKEN with a "write" token -
# the key icon in the left sidebar - so login() doesn't need to prompt you.
HF_USERNAME = "abhishek15016"  # leave blank to skip the Hub entirely
PUSH_TO_HUB = bool(HF_USERNAME)
PRIVATE_REPO = True
HF_MODEL_PREFIX = "qwen2.5-0.5b-genai-agentic"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if PUSH_TO_HUB:
    login(token=HF_TOKEN or None)  # prompts interactively if HF_TOKEN isn't set

STAGE1_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage1-non-instruction" if HF_USERNAME else ""
STAGE2_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage2-sft" if HF_USERNAME else ""
STAGE3_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage3-dpo" if HF_USERNAME else ""

def push_adapter(model, tokenizer, repo_id, stage_name):
    if not (PUSH_TO_HUB and repo_id):
        print(f"Skipping Hub push for {stage_name} (set HF_USERNAME above to enable).")
        return
    model.push_to_hub(repo_id, token=HF_TOKEN or None, private=PRIVATE_REPO)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN or None, private=PRIVATE_REPO)
    print(f"Pushed {stage_name} adapter to https://huggingface.co/{repo_id}")


## Step A - Load the model

By default we continue from the Stage 1 non-instruction adapter: local disk first, then the Hub, then the untouched base model. Set `CONTINUE_FROM_STAGE1 = False` to always start instruction tuning straight from the untouched base model.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
BASE_MODEL_NAME = "unsloth/Qwen2.5-0.5B-bnb-4bit"
STAGE1_LOCAL_DIR = "models/non_instruction_adapter"
CONTINUE_FROM_STAGE1 = True

import os
if not CONTINUE_FROM_STAGE1:
    start_from = BASE_MODEL_NAME
elif os.path.exists(STAGE1_LOCAL_DIR):
    start_from = STAGE1_LOCAL_DIR
elif STAGE1_HF_REPO:
    start_from = STAGE1_HF_REPO
    print(f"No local Stage 1 adapter - pulling from the Hub: {STAGE1_HF_REPO}")
else:
    start_from = BASE_MODEL_NAME
    print("No local or Hub Stage 1 adapter found - starting from the untouched base model instead.")
print("Loading from:", start_from)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=start_from,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)


## Step B - Apply the ChatML template

Qwen2.5 was pretrained on ChatML-style special tokens (`<|im_start|>` / `<|im_end|>`), so we make that explicit on the tokenizer rather than relying on a template being predefined.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

def formatting_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["messages"]
    ]
    return {"text": texts}


## Step C - Load and format the instruction dataset

In [ ]:
from datasets import load_dataset

instruction_ds = load_dataset("json", data_files="data/instruction_dataset.jsonl", split="train")
print(f"{len(instruction_ds)} instruction-response examples")

instruction_ds = instruction_ds.map(formatting_func, batched=True)
print(instruction_ds[0]["text"][:500])

split = instruction_ds.train_test_split(test_size=0.1, seed=3407)
train_ds, eval_ds = split["train"], split["test"]
len(train_ds), len(eval_ds)


## Step D - Apply LoRA

In [ ]:
if not hasattr(model, "peft_config"):
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
model.print_trainable_parameters()


## Step E - Train (loss masked to assistant responses only)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir="outputs/stage2_instruction",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
)

# Mask the loss so the model only learns to predict assistant turns, not the
# system/user prompt tokens - standard best practice for instruction tuning.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


In [ ]:
# Take a screenshot of this cell's output for your README ("Training screenshots or logs").
train_result = trainer.train()
train_result.metrics


## Step F - Save the Stage 2 (SFT) adapter

In [ ]:
SAVE_DIR = "models/sft_adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved SFT adapter to", SAVE_DIR)


In [ ]:
push_adapter(model, tokenizer, STAGE2_HF_REPO, "Stage 2 (SFT)")


## Step G - Compare base model vs. instruction fine-tuned model (Step 7 of the assignment)

We reload a fresh pristine base model instance so the comparison is a clean, paired before/after on identical prompts and decoding settings.

In [ ]:
FastLanguageModel.for_inference(model)

def generate_chat(active_model, active_tokenizer, question, max_new_tokens=220):
    convo = [
        {"role": "system", "content": "You are a friendly expert tutor in Generative AI and Agentic AI. Explain concepts in depth with intuitive analogies, practical production insight, and interview-ready takeaways, in a clear and engaging style."},
        {"role": "user", "content": question},
    ]
    prompt = active_tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=True)
    inputs = active_tokenizer(prompt, return_tensors="pt").to(active_model.device)
    out = active_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=active_tokenizer.eos_token_id,
    )
    text = active_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()

sft_answers = [generate_chat(model, tokenizer, q) for q in EVAL_QUESTIONS]
for q, a in zip(EVAL_QUESTIONS, sft_answers):
    print("Q:", q)
    print("A (SFT):", a[:300])
    print("-" * 80)


In [ ]:
# Fresh pristine base model for a clean paired comparison (separate from the
# Stage-1-adapted `model` object above).
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

from src.eval_questions import clean_completion

def generate_raw(question, max_new_tokens=180):
    prompt = f"Question: {question}\nAnswer:"
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    out = base_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,
        top_p=0.9, repetition_penalty=1.3, pad_token_id=base_tokenizer.eos_token_id,
    )
    text = base_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return clean_completion(text, question)

base_answers = [generate_raw(q) for q in EVAL_QUESTIONS]
del base_model
torch.cuda.empty_cache()


In [ ]:
rows = []
for q, base_a, sft_a in zip(EVAL_QUESTIONS, base_answers, sft_answers):
    winner, reason = judge_pair(q, "Fine-Tuned", sft_a, "Fine-Tuned", sft_a)  # placeholder, overwritten below
    winner, reason = judge_pair(q, "Base", base_a, "Fine-Tuned", sft_a)
    rows.append((q, base_a, sft_a, winner, reason))

table = markdown_table(
    ["Question", "Base Model Answer", "Fine-Tuned Model Answer", "Which is Better?", "Reason"],
    rows,
)

report = f"""# Base vs. Instruction Fine-Tuned (SFT) Model Comparison

**Base model:** `{BASE_MODEL_NAME}` (no fine-tuning)
**Fine-tuned model:** Stage 1 (non-instruction) + Stage 2 (instruction/SFT) on `data/instruction_dataset.jsonl`

{table}

## Evaluation criteria
Correctness, domain accuracy, clarity, safety, helpfulness, less-generic
responses, better domain-specific behavior (per assignment Step 7).

## Summary
Across the 10 held-out evaluation questions, the instruction fine-tuned model
consistently answers in the trained domain voice - structured explanations,
correct GenAI/Agentic-AI terminology, and an interview-ready framing - instead
of the generic or off-topic completions produced by the untouched base model.

*Auto-generated by `notebooks/instruction_finetuning.ipynb`. The "Which is
Better?" / "Reason" columns are a heuristic first draft (see
`src/eval_questions.py`) - review before submission.*
"""

with open("reports/sft_model_comparison.md", "w", encoding="utf-8") as f:
    f.write(report)

print(report)


---
### Next: `notebooks/dpo_alignment.ipynb`

That notebook loads `models/sft_adapter/` (or pulls `STAGE2_HF_REPO` from the
Hugging Face Hub if this ran in a separate Colab session) and runs DPO
preference alignment on `data/preference_dataset.jsonl`.